# Thermal-averaged reaction rates from S(E) or σ(E)

This notebook converts a nuclear cross-section — supplied either as an
astrophysical S-factor `S(E)` or directly as `σ(E)` — into a **primat-format
thermonuclear rate table**, with a Monte-Carlo-propagated 1σ uncertainty
column and the correct detailed-balance header.

It is the Python replacement for the Mathematica notebook
`Thermal-Average.nb`, which lived outside this repository and is superseded by
this file.

## Physics

For charged particles the cross-section is factorised into the slowly-varying
astrophysical S-factor and the Coulomb (Gamow) penetration factor:

$$\sigma(E) = \frac{S(E)}{E}\,e^{-2\pi\eta},
\qquad \eta = \alpha_{\rm FS} Z_1 Z_2 \sqrt{\frac{\mu c^2}{2E}} $$

The thermal average over a Maxwell–Boltzmann distribution at temperature $T$ is

$$N_A\langle\sigma v\rangle
  = N_A \left(\frac{8}{\pi\mu}\right)^{1/2} (k_BT)^{-3/2}
    \int_0^\infty \sigma(E)\, E\, e^{-E/k_BT}\, {\rm d}E $$

This is the energy-space form of the velocity integral used in the Mathematica
notebook; the two are identical under $E = \tfrac12\mu v^2$, but the energy
form is better conditioned near the Gamow peak.

Reference: Pitrou, Coc, Uzan & Vangioni, *Physics Reports* **04** (2018) 005
(`biblio/Pitrou_etal_PhysReptArxivVersion.pdf`), nuclear-rates section.

## Units

| Quantity | Unit |
|---|---|
| Energy `E` | MeV |
| S-factor `S(E)` | MeV·barn |
| Cross-section `σ(E)` | barn |
| Temperature `T9` | 10⁹ K |
| Output `N_A⟨σv⟩` | cm³ mol⁻¹ s⁻¹ |

## How to use this notebook

**Edit §3 only.** Everything else is machinery. In §3 you declare the
reaction, a reference label for the output filename, whether you are giving
`S(E)` or `σ(E)`, the function itself, its parameter vector and covariance,
and where to write the result. Then run all cells.

All nuclide data — masses, charges, spins, Q-values, detailed-balance
coefficients — is read from primat itself, so this notebook cannot drift from
the solver's own nuclear data.

## §2 — Nuclide data and constants (from primat)

Nothing here is hard-coded. `PRIMATConfig` reads
`primat/data/csv/nuclides.csv` (generated offline by
`generate_rates/nuclide_table.py` from the NUBASE2020 evaluation
`nubase_4.mas20.txt`), giving `(N, Z)`, mass excess in keV, and spin for
every nuclide in primat's reaction catalog.

In [ ]:
import sys
from pathlib import Path

import numpy as np

# The notebook lives in generate_rates/; primat is importable from the repo root.
REPO = Path.cwd().parent if Path.cwd().name == "generate_rates" else Path.cwd()
sys.path.insert(0, str(REPO))

from primat.config import PRIMATConfig
from primat.constants import CONST

cfg = PRIMATConfig()

# --- Unit conversions and constants, all taken from primat ------------------
ALPHA_FS = CONST.alphaem              # fine-structure constant (dimensionless)
M_U_MEV  = cfg.ma                     # atomic mass unit          [MeV]
M_E_MEV  = cfg.me                     # electron mass             [MeV]
BARN_CM2 = 1.0e-24                    # 1 barn in cm^2 (definition of the barn)

# k_B * 1e9 K expressed in MeV, i.e. kT[MeV] = MEV_PER_GK * T9.  cfg.kB is in
# erg/K and cfg.MeV is 1 MeV in erg, so the ratio converts erg -> MeV.
MEV_PER_GK = cfg.kB * 1.0e9 / cfg.MeV

# Avogadro's number is *derived*, not typed in: it is the reciprocal of the
# atomic mass unit expressed in grams (m_u[MeV] * erg/MeV / c^2).
N_A = 1.0 / (M_U_MEV * cfg.MeV / cfg.clight**2)   # [mol^-1]


def nuclear_mass_MeV(name):
    """Rest-mass energy of a nuclide, in MeV.

    This is the *nuclear* mass (bare nucleus), not the atomic mass: the Z bound
    electrons are subtracted.  It is built exactly the way
    ``primat.network_data.compute_detailed_balance_coefficients`` builds it, so
    the reduced mass used here is consistent with the reverse-rate coefficients
    written into the output file's header:

        M = A * m_u + Delta - Z * m_e

    with ``Delta`` the mass excess from ``nuclides.csv`` (stored in keV, hence
    the 1e-3 conversion to MeV).

    Args:
        name: primat nuclide key, e.g. ``"n"``, ``"p"``, ``"H2"``, ``"He4"``.

    Returns:
        Rest-mass energy in MeV.

    Example:
        >>> round(nuclear_mass_MeV("H2"), 4)     # deuteron
        1875.6128
    """
    N, Z = cfg.Nuclides[name]
    A = N + Z
    return A * M_U_MEV + cfg.NuclExcessMass[name] * 1.0e-3 - Z * M_E_MEV


def charge(name):
    """Atomic number Z of a nuclide, from primat's ``Nuclides`` table.

    Z enters the Gamow penetration factor as the product Z1*Z2; it is zero for
    the neutron, which is why neutron-induced reactions have no Coulomb barrier.

    Args:
        name: primat nuclide key, e.g. ``"He4"``.

    Returns:
        Atomic number (int).

    Example:
        >>> charge("He4"), charge("n")
        (2, 0)
    """
    return cfg.Nuclides[name][1]


def reduced_mass_MeV(n1, n2):
    """Reduced rest-mass energy mu*c^2 of the entrance channel, in MeV.

    The thermal average is an integral over the *relative* kinetic energy of
    the two reactants, whose inertia is the reduced mass
    ``mu = m1 m2 / (m1 + m2)``.  Working with mu*c^2 in MeV keeps every energy
    in the notebook in the same unit.

    Args:
        n1, n2: primat nuclide keys of the two reactants.

    Returns:
        mu*c^2 in MeV.

    Example:
        >>> round(reduced_mass_MeV("H2", "H2"), 4)   # half the deuteron mass
        937.8064
    """
    m1, m2 = nuclear_mass_MeV(n1), nuclear_mass_MeV(n2)
    return m1 * m2 / (m1 + m2)


print(f"N_A        = {N_A:.6e} mol^-1        (expect 6.022141e+23)")
print(f"MEV_PER_GK = {MEV_PER_GK:.7f} MeV/T9  (expect 0.0861733)")
print(f"M(d)       = {nuclear_mass_MeV('H2'):.4f} MeV   (expect 1875.6128)")
print(f"mu(d,d)    = {reduced_mass_MeV('H2', 'H2'):.4f} MeV   (expect 937.8064)")

## §3 — USER INPUT (edit this section, and nothing else)

`cross_section(E_MeV, theta)` must be vectorized in `E_MeV` and take a
1-D parameter vector `theta`, so that the Monte Carlo of §6 can resample it.
Return **MeV·barn** when `MODE="S"`, **barn** when `MODE="sigma"`.

Use `MODE="sigma"` for neutron-induced reactions (Z₁Z₂ = 0), where the
S-factor factorisation buys you nothing.

Instead of writing a formula you can read a tabulated cross-section from a
file with `cross_section = from_table(path, kind="S")` (see §4); `theta[0]`
then acts as a log-normal overall normalisation, so an overall systematic
uncertainty is set with `THETA0 = np.array([0.0])` and `COV = [[0.05**2]]`
for 5%.

Uncertainties: give `COV`, the covariance matrix of `theta`, for a Gaussian
Monte Carlo. Set `COV = None` for no uncertainty (the error column becomes
1.0). For a non-Gaussian prior, set `COV = None` and provide
`SAMPLE_THETA = lambda rng, n: <(n, len(THETA0)) array>` instead.

In [ ]:
# ===========================================================================
#  USER INPUT
# ===========================================================================
# The reaction, by primat's reaction name (the directory name under
# primat/data/nuclear/tables/).  Species short names are joined by "_", and
# reactants are separated from products by "__":  "d_d__t_p" is d + d -> t + p.
REACTION = "d_d__t_p"

# Reference label; becomes the filename suffix and appears in the file header.
REF = "Mathematica-Sddp"

# "S"     -> cross_section returns the astrophysical S-factor in MeV*barn
# "sigma" -> cross_section returns the cross-section directly, in barn
MODE = "S"


def cross_section(E_MeV, theta):
    """S-factor of d + d -> t + p, in MeV*barn.

    Worked example: the polynomial fit used in the Mathematica notebook
    ``Thermal-Average.nb``,
    ``S(E) = 0.05520 + 0.2151 E - 0.02555 E^2`` with E in MeV.  Replace this
    body with your own parametrisation.

    Args:
        E_MeV: centre-of-mass kinetic energy [MeV]; may be an array.
        theta: 1-D parameter vector; here the three polynomial coefficients.

    Returns:
        S(E) in MeV*barn, same shape as ``E_MeV``.

    Example:
        >>> float(cross_section(np.array([0.1]), THETA0))
        0.0765...
    """
    return theta[0] + theta[1] * E_MeV + theta[2] * E_MeV**2


# Central parameter values.
THETA0 = np.array([0.05520, 0.2151, -0.02555])

# Covariance matrix of theta, or None for "no uncertainty".  The Mathematica
# notebook quoted no uncertainty on this fit, so the worked example uses a
# 2% fully-correlated normalisation error as an illustration: a 2% error on
# the constant term alone would be inconsistent, so it is applied by scaling
# the whole vector (see SAMPLE_THETA below for the general case).
COV = np.diag((0.02 * np.abs(THETA0))**2)

# Optional non-Gaussian sampler: SAMPLE_THETA(rng, n) -> (n, len(THETA0)).
# Leave as None to use the Gaussian COV above.
SAMPLE_THETA = None

N_MC = 300          # Monte-Carlo samples; 300 is ample for a 16/84 percentile
SEED = 20260728     # fixed seed so the written table is reproducible

# Where to write.  The default is an untracked overlay directory, so this
# notebook never touches the shipped primat/data/ tree.  Its layout is exactly
# what primat's `user_nuclear_dir` parameter expects, so a run can pick the new
# table up with  user_nuclear_dir="<REPO>/generate_rates/rate_tables_out"
# (see §8).  Point OUTDIR at primat/data/nuclear/tables only if you really do
# intend to modify the shipped data.
OUTDIR = REPO / "generate_rates" / "rate_tables_out" / "tables"

OVERWRITE = False   # refuse to clobber an existing file unless True
# ===========================================================================

## §4 — Uniform cross-section interface

Everything the user can supply — an S-factor formula, a σ(E) formula, or a
tabulated file — is collapsed here into a single function
`sigma_of_E_cm2(E_MeV, theta)` returning cm². No code below this point
branches on `MODE`.

In [ ]:
from primat.network_data import reaction_species

# --- Resolve the reaction against primat's catalog --------------------------
try:
    REACTANTS, PRODUCTS = reaction_species(REACTION)
except Exception as exc:
    # A bad reaction name is by far the most common user error, so say exactly
    # what went wrong and offer the near-misses from the shipped tables tree.
    known = sorted(p.name for p in (REPO / "primat/data/nuclear/tables").iterdir()
                   if p.is_dir())
    stem = REACTION.split("__")[0]
    close = [k for k in known if k.startswith(stem[:3])]
    raise ValueError(
        f"REACTION={REACTION!r} is not a reaction primat knows: {exc}\n"
        f"Did you mean one of: {close[:10]}\n"
        "If the *nuclide* itself is missing from primat/data/csv/nuclides.csv, "
        "extend the catalog with generate_rates/nuclide_table.py first."
    ) from exc

if len(REACTANTS) != 2:
    raise ValueError(
        f"{REACTION} has {len(REACTANTS)} reactants ({REACTANTS}); the "
        "two-body thermal average implemented here needs exactly 2."
    )

MU_MEV = reduced_mass_MeV(*REACTANTS)          # entrance-channel reduced mass
Z1Z2 = charge(REACTANTS[0]) * charge(REACTANTS[1])

if MODE == "S" and Z1Z2 == 0:
    print(f"WARNING: {REACTION} has Z1*Z2 = 0, so there is no Coulomb barrier "
          "and the Gamow factor is 1.  S(E) then means nothing more than "
          "sigma(E)*E; MODE='sigma' is the natural choice.")

# Counter for clipped unphysical (negative) cross-sections; a badly
# extrapolated polynomial must not fail silently.
NEGATIVE_SIGMA_COUNT = [0]


def gamow_exponent(E_MeV, mu_MeV, z1z2):
    """The exponent 2*pi*eta of the Coulomb penetration factor.

    The Sommerfeld parameter is eta = Z1 Z2 alpha c / v; with the relative
    velocity written in terms of the centre-of-mass energy,
    v = sqrt(2E/mu), this becomes eta = Z1 Z2 alpha sqrt(mu c^2 / 2E).  The
    tunnelling probability through the Coulomb barrier is exp(-2 pi eta),
    which is what suppresses charged-particle rates at low temperature.

    Args:
        E_MeV: centre-of-mass energy [MeV], array-like.
        mu_MeV: reduced rest-mass energy mu*c^2 [MeV].
        z1z2: product of the reactants' atomic numbers (0 for neutrons).

    Returns:
        2*pi*eta, dimensionless, same shape as ``E_MeV``.

    Example:
        >>> float(gamow_exponent(np.array([0.1]), 937.8064, 1))
        9.93...
    """
    return 2.0 * np.pi * ALPHA_FS * z1z2 * np.sqrt(mu_MeV / (2.0 * E_MeV))


def from_table(path, kind="S"):
    """Build a ``cross_section``-shaped callable from a tabulated file.

    Reads a whitespace- or comma-separated file whose first column is the
    energy in MeV and whose second column is S(E) in MeV*barn (``kind="S"``)
    or sigma(E) in barn (``kind="sigma"``); further columns are ignored.
    Interpolation is linear in log-log, which is the right choice for
    quantities spanning decades, and is clamped (not extrapolated) outside
    the tabulated range — extrapolating a measured cross-section is a
    physics decision the user should make deliberately, not a side effect.

    ``theta[0]`` acts as a log-normal overall normalisation
    (sigma -> sigma * exp(theta[0])), so the Monte Carlo of §6 propagates an
    overall systematic uncertainty with ``THETA0 = np.array([0.0])`` and
    ``COV = [[rel_err**2]]``.

    Args:
        path: path to the two-or-more-column table.
        kind: ``"S"`` or ``"sigma"``, matching ``MODE``.

    Returns:
        A callable ``f(E_MeV, theta)`` with the same contract as the
        hand-written ``cross_section``.

    Example:
        >>> cross_section = from_table("my_sfactor.dat", kind="S")
        >>> THETA0 = np.array([0.0]); COV = np.array([[0.05**2]])
    """
    data = np.loadtxt(path, delimiter=None, comments="#", ndmin=2)
    E_tab, y_tab = data[:, 0], data[:, 1]
    order = np.argsort(E_tab)
    logE, logy = np.log(E_tab[order]), np.log(y_tab[order])

    def f(E_MeV, theta):
        # np.interp clamps to the end values outside [E_tab[0], E_tab[-1]].
        y = np.exp(np.interp(np.log(E_MeV), logE, logy))
        return y * np.exp(theta[0])

    f.kind = kind
    return f


def sigma_of_E_cm2(E_MeV, theta):
    """Cross-section in cm^2, whatever form the user supplied it in.

    This is the single interface the thermal-average kernel of §5 sees.  For
    ``MODE="S"`` it applies the Gamow factorisation
    ``sigma = S(E)/E * exp(-2 pi eta)``; for ``MODE="sigma"`` it merely
    converts barn to cm^2.  Negative values (an over-extrapolated polynomial,
    say) are unphysical and are clipped to zero, with a running count kept in
    ``NEGATIVE_SIGMA_COUNT`` so the clipping is reported rather than hidden.

    Args:
        E_MeV: centre-of-mass energy [MeV], array-like.
        theta: parameter vector passed straight through to ``cross_section``.

    Returns:
        sigma(E) in cm^2, same shape as ``E_MeV``.

    Example:
        >>> float(sigma_of_E_cm2(np.array([0.1]), THETA0))    # d+d at 100 keV
        3.7e-26...
    """
    y = np.asarray(cross_section(E_MeV, theta), dtype=float)
    if MODE == "S":
        sigma_barn = y / E_MeV * np.exp(-gamow_exponent(E_MeV, MU_MEV, Z1Z2))
    elif MODE == "sigma":
        sigma_barn = y
    else:
        raise ValueError(f"MODE must be 'S' or 'sigma', got {MODE!r}")
    n_neg = int(np.count_nonzero(sigma_barn < 0.0))
    if n_neg:
        NEGATIVE_SIGMA_COUNT[0] += n_neg
        sigma_barn = np.clip(sigma_barn, 0.0, None)
    return sigma_barn * BARN_CM2


print(f"{REACTION}: {' + '.join(REACTANTS)} -> {' + '.join(PRODUCTS)}")
print(f"  mu = {MU_MEV:.4f} MeV, Z1*Z2 = {Z1Z2}")
print(f"  2*pi*eta at E = 100 keV: "
      f"{gamow_exponent(np.array([0.1]), MU_MEV, Z1Z2)[0]:.4f}")
print(f"  sigma(100 keV) = {sigma_of_E_cm2(np.array([0.1]), THETA0)[0]:.4e} cm^2")